# Figure S10 — PIP agreement: combined (estimated) mixture vs. true-weight reference mixture (data processing)

From the existing ADSP + UKB meta-analysis fits (`12` `[finemap_sweep]`, `ADSP_UKB_fine`, total n = 2,000; ADSP shares 20 / 50 / 80%). Same meta-analysis z, baseline SuSiE-RSS with

- `gen_refpanel`: the ADSP and UKB reference LD combined with the true sample-size weight, and
- `mix_refpanel`: the combined fit with weights estimated in the same `susie_rss(R = list(...))` call.

`gen_insample` in these files is built from the same full panels (not the study samples), so it equals `gen_refpanel`; the first check cell confirms this. Word the panel as reference LD with true vs. estimated weights, not in-sample LD.

**Outputs:** `figure_data/FigS10.Combined_vs_True_Weight_PIP_Agreement.rds` (per replicate: PIP correlation, max |ΔPIP|, credible-set counts, true and estimated ADSP weight), `FigS10.Combined_vs_True_Weight_PIP_Examples.rds` (PIPs of the example replicates, variants in a credible set of either fit; r over all variants)

In [1]:
source("../../R/utils.R")

res_dir <- "/hpc/users/sunh17/haochen/rss_qc/2026_output/ADSP_UKB_fine/result"   # 12 [finemap_sweep]
props   <- c("0.20", "0.50", "0.80")      # ADSP shares with both studies present
n_show  <- 6                              # example replicates (2 per ADSP share)

files <- list.files(res_dir, pattern = "\\.rds$", recursive = TRUE, full.names = TRUE)
files <- files[str_extract(files, "(?<=prop_)[0-9.]+") %in% props]
length(files)

## random subset, spread evenly over the settings (h2g x causal x ADSP share folders); fixed seed
n_files <- 1500
set.seed(2026)
by_setting <- split(files, dirname(files))
per_set    <- ceiling(n_files / length(by_setting))
files <- unlist(lapply(by_setting, function(f) if (length(f) > per_set) sample(f, per_set) else f), use.names = FALSE)
c(settings = length(by_setting), per_setting = per_set, files_used = length(files))

[1] 13392

settings per_setting  files_used 
         27          56        1512

## Agreement across replicates

In [2]:
## ---- per replicate: reference mixture with true weights vs. estimated combined mixture ----
##   gen_refpanel : w R_ADSP + (1 - w) R_UKB, true sample-size weight w (full-panel LD)
##   mix_refpanel : susie_rss(R = list(R_ADSP, R_UKB)), weights estimated in the same call
##   gen_insample : meant as in-sample LD, but built from the same full panels -> checked to equal gen_refpanel
agree <- map_dfr(files, function(f) {
  r <- readRDS(f)
  if (!identical(r$status, "ok")) return(NULL)
  a <- r$result_list$gen_refpanel; b <- r$result_list$mix_refpanel; g <- r$result_list$gen_insample
  if (is.null(a) || is.null(b)) return(NULL)
  tibble(file = f, h2g = as.numeric(r$h2g), causal = as.integer(r$causal), prop = r$prop,
         w = r$w, omega_ADSP = r$omega_weights[1],
         pip_cor = suppressWarnings(cor(a$pip, b$pip)), max_abs_diff = max(abs(a$pip - b$pip)),
         n_cs_true_w = length(a$sets$cs), n_cs_combined = length(b$sets$cs),
         max_pip = max(a$pip),
         insample_equals_true_w = !is.null(g) && isTRUE(all.equal(as.numeric(g$pip), as.numeric(a$pip))))
})

## check: gen_insample is the true-weight reference mixture
agree %>% count(insample_equals_true_w)

## agreement by ADSP share
agree %>% group_by(prop) %>%
  summarise(n = n(), median_pip_cor = median(pip_cor, na.rm = TRUE), q25 = quantile(pip_cor, .25, na.rm = TRUE),
            median_max_abs_diff = median(max_abs_diff), same_n_cs = mean(n_cs_true_w == n_cs_combined), .groups = "drop")

save_data(select(agree, -file) %>% mutate(file = basename(agree$file)), "FigS10.Combined_vs_True_Weight_PIP_Agreement")

insample_equals_true_w,n
<lgl>,<int>
TRUE,1512


prop,n,median_pip_cor,q25,median_max_abs_diff,same_n_cs
<chr>,<int>,<dbl>,<dbl>,<dbl>,<dbl>
0.20,504,0.9999963,0.9999812,1.487359e-03,0.9900794
0.50,504,1.0000000,0.9999966,2.040932e-05,0.9980159
0.80,504,0.9999971,0.9999847,1.269515e-03,0.9920635


## Example replicates

In [3]:
## ---- example replicates: per ADSP share, the two with the strongest signal ----------------
pick <- agree %>% filter(n_cs_true_w > 0) %>%
  group_by(prop) %>% slice_max(max_pip, n = n_show / length(props), with_ties = FALSE) %>% ungroup()

pips <- map_dfr(seq_len(nrow(pick)), function(i) {
  r <- readRDS(pick$file[i])
  a <- r$result_list$gen_refpanel; b <- r$result_list$mix_refpanel
  tibble(pip_true_w = a$pip, pip_combined = b$pip,
         in_cs = seq_along(a$pip) %in% c(unlist(a$sets$cs), unlist(b$sets$cs)),   # in a CS of either fit
         causal_variant = seq_along(a$pip) %in% r$true_vars,
         panel = sprintf("ADSP %.0f%%, %d causal, replicate %s", 100 * as.numeric(pick$prop[i]), pick$causal[i],
                         str_extract(basename(pick$file[i]), "(?<=sample_)[0-9]+")),
         label = sprintf("r = %.3f (all variants)", pick$pip_cor[i]))
}) %>%
  filter(in_cs)                                    # plot only variants in a credible set of either fit

pips %>% count(panel)                              # points per panel

save_data(pips, "FigS10.Combined_vs_True_Weight_PIP_Examples")

panel,n
<chr>,<int>
"ADSP 20%, 2 causal, replicate 45",8
"ADSP 20%, 2 causal, replicate 489",4
"ADSP 50%, 1 causal, replicate 436",1
"ADSP 50%, 3 causal, replicate 443",27
"ADSP 80%, 2 causal, replicate 104",14
"ADSP 80%, 3 causal, replicate 244",3
